<a href="https://colab.research.google.com/github/VintaBytes/Ciencia-de-datos/blob/main/Ciencia%20De%20Datos%20Con%20Python/CuadernosColab/CienciaDeDatos_Cap%C3%ADtulo022_Validaci%C3%B3n_posterior_a_la_limpieza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Capítulo 22 — Validación posterior a la limpieza

## Breve repaso

En el trabajo anterior construimos un flujo de limpieza encadenado. Partimos de un dataset crudo, creamos una copia de trabajo, estandarizamos nombres de columnas, convertimos tipos de datos, revisamos valores categóricos problemáticos, creamos columnas calculadas, agregamos una columna de validación y construimos una versión preparada del `DataFrame`.

Ese recorrido mostró que la limpieza de datos no suele resolverse con una sola instrucción. Preparar un dataset implica aplicar varias transformaciones en un orden razonable y verificar el resultado de cada etapa.

En este capítulo vamos a concentrarnos en el paso posterior: validar la limpieza realizada.

Validar no significa demostrar que el dataset quedó perfecto. Significa revisar si la versión preparada quedó en mejores condiciones para el análisis, qué problemas fueron tratados, qué problemas siguen presentes y qué decisiones deberían quedar documentadas.

La validación posterior es una especie de control de calidad. Nos ayuda a no asumir que el dataset está listo solo porque aplicamos varias transformaciones.

A lo largo del capítulo vamos a revisar dimensiones, tipos de datos, valores faltantes, duplicados, categorías problemáticas, rangos numéricos, fechas válidas y consistencia interna.

Al finalizar este capítulo deberías poder:

- Comprender por qué es necesario validar después de limpiar.
- Comparar el dataset original con el dataset preparado.
- Revisar tipos de datos finales.
- Revisar valores faltantes después de la limpieza.
- Detectar si quedan duplicados.
- Revisar categorías problemáticas pendientes.
- Revisar rangos numéricos y fechas válidas.
- Validar relaciones internas entre columnas.
- Construir una síntesis del estado final del dataset.
- Reconocer qué problemas quedan pendientes y documentarlos.

## Dataset de trabajo

Para validar una limpieza necesitamos contar con dos versiones del dataset:

```text
df_original   → el dataset tal como fue cargado desde el archivo
df_preparado  → una versión transformada y más lista para analizar
```

En este capítulo vamos a reconstruir una versión preparada aplicando el mismo flujo general que trabajamos antes. No vamos a explicar cada transformación desde cero, porque esas herramientas ya fueron desarrolladas. El foco estará puesto en validar el resultado final.

In [51]:
import kagglehub
import pandas as pd
import os

ruta_dataset = kagglehub.dataset_download(
    "ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training"
)

archivos = os.listdir(ruta_dataset)

archivos_csv = [
    archivo for archivo in archivos
    if archivo.endswith(".csv")
]

ruta_csv = os.path.join(ruta_dataset, archivos_csv[0])

df_original = pd.read_csv(ruta_csv)

df_original.head()

Using Colab cache for faster access to the 'cafe-sales-dirty-data-for-cleaning-training' dataset.


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


Ahora tenemos cargado el dataset original.

A continuación vamos a construir una versión preparada usando una secuencia compacta de transformaciones: renombrar columnas, convertir tipos, tratar algunos valores categóricos problemáticos y crear columnas útiles para validar y analizar.

In [52]:
df_limpieza = df_original.copy()

df_limpieza = df_limpieza.rename(columns={
    "Transaction ID": "transaction_id",
    "Item": "item",
    "Quantity": "quantity",
    "Price Per Unit": "price_per_unit",
    "Total Spent": "total_spent",
    "Payment Method": "payment_method",
    "Location": "location",
    "Transaction Date": "transaction_date"
})

columnas_numericas = [
    "quantity",
    "price_per_unit",
    "total_spent"
]

for columna in columnas_numericas:
    df_limpieza[columna] = pd.to_numeric(
        df_limpieza[columna],
        errors="coerce"
    )

df_limpieza["transaction_date"] = pd.to_datetime(
    df_limpieza["transaction_date"],
    errors="coerce"
)

columnas_categoricas = [
    "item",
    "payment_method",
    "location"
]

for columna in columnas_categoricas:
    df_limpieza[columna] = df_limpieza[columna].replace({
        "ERROR": "UNKNOWN"
    })

df_limpieza["calculated_total"] = (
    df_limpieza["quantity"] * df_limpieza["price_per_unit"]
)

df_limpieza["estado_total"] = "sin_datos_suficientes"

condicion_datos_completos = (
    df_limpieza["quantity"].notna()
    & df_limpieza["price_per_unit"].notna()
    & df_limpieza["total_spent"].notna()
)

condicion_coincide = (
    condicion_datos_completos
    & (df_limpieza["total_spent"] == df_limpieza["calculated_total"])
)

condicion_no_coincide = (
    condicion_datos_completos
    & (df_limpieza["total_spent"] != df_limpieza["calculated_total"])
)

df_limpieza.loc[condicion_coincide, "estado_total"] = "coincide"
df_limpieza.loc[condicion_no_coincide, "estado_total"] = "no_coincide"

df_limpieza["year"] = df_limpieza["transaction_date"].dt.year
df_limpieza["month"] = df_limpieza["transaction_date"].dt.month
df_limpieza["day_of_week"] = df_limpieza["transaction_date"].dt.day_name()

columnas_finales = [
    "transaction_id",
    "item",
    "quantity",
    "price_per_unit",
    "total_spent",
    "calculated_total",
    "estado_total",
    "payment_method",
    "location",
    "transaction_date",
    "year",
    "month",
    "day_of_week"
]

df_preparado = df_limpieza[columnas_finales].copy()

df_preparado.head()

,transaction_id,item,quantity,price_per_unit,total_spent,calculated_total,estado_total,payment_method,location,transaction_date,year,month,day_of_week
0,TXN_1961373,Coffee,2.0,2.0,4.0,4.0,coincide,Credit Card,Takeaway,2023-09-08,2023.0,9.0,Friday
1,TXN_4977031,Cake,4.0,3.0,12.0,12.0,coincide,Cash,In-store,2023-05-16,2023.0,5.0,Tuesday
2,TXN_4271903,Cookie,4.0,1.0,NaN,4.0,sin_datos_suficientes,Credit Card,In-store,2023-07-19,2023.0,7.0,Wednesday
3,TXN_7034554,Salad,2.0,5.0,10.0,10.0,coincide,UNKNOWN,UNKNOWN,2023-04-27,2023.0,4.0,Thursday
4,TXN_3160411,Coffee,2.0,2.0,4.0,4.0,coincide,Digital Wallet,In-store,2023-06-11,2023.0,6.0,Sunday


Ya tenemos una versión preparada del dataset.

A partir de ahora, el objetivo no será seguir transformando datos, sino validar el estado de `df_preparado` y compararlo cuando sea necesario con `df_original`.

## Comparar dimensiones antes y después

Una primera validación consiste en comparar las dimensiones del dataset original y del dataset preparado.

Esto permite saber si durante el proceso cambiaron la cantidad de filas o columnas.

En nuestro flujo de limpieza no eliminamos filas. Sí agregamos columnas nuevas y seleccionamos una estructura final diferente. Por eso esperamos que la cantidad de filas se mantenga, pero que la cantidad de columnas cambie.

In [53]:
print("Dimensiones del dataset original:")
print(df_original.shape)

print()

print("Dimensiones del dataset preparado:")
print(df_preparado.shape)

Dimensiones del dataset original:
(10000, 8)

Dimensiones del dataset preparado:
(10000, 13)


La comparación entre dimensiones nos permite responder dos preguntas:

```text
¿Se conservaron todas las filas?
¿Cambió la cantidad de columnas?
```

Si la cantidad de filas se mantiene, significa que el flujo aplicado no eliminó registros. Esto es importante porque algunas decisiones de limpieza, como `dropna()` o `drop_duplicates()`, pueden reducir el dataset.

En este caso, el cambio principal está en la estructura de columnas. El dataset preparado conserva las variables principales, pero también incluye columnas nuevas como `calculated_total`, `estado_total`, `year`, `month` y `day_of_week`.

Comparar dimensiones es una validación simple, pero muy útil. Nos ayuda a detectar cambios estructurales inesperados.


## Revisar tipos de datos finales

Después de preparar el dataset, necesitamos revisar si las columnas quedaron con tipos de datos adecuados.

En el dataset original, todas las columnas habían sido cargadas como `object`. Eso incluía columnas que deberían ser numéricas y una columna que debería ser temporal.

En el dataset preparado esperamos observar una estructura más adecuada:

```text
quantity, price_per_unit, total_spent, calculated_total → columnas numéricas
transaction_date                                      → columna temporal
item, payment_method, location, estado_total           → columnas categóricas o textuales
```

Podemos revisar los tipos de datos finales con `info()`.

In [54]:
df_preparado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    10000 non-null  object        
 1   item              9667 non-null   object        
 2   quantity          9521 non-null   float64       
 3   price_per_unit    9467 non-null   float64       
 4   total_spent       9498 non-null   float64       
 5   calculated_total  9006 non-null   float64       
 6   estado_total      10000 non-null  object        
 7   payment_method    7421 non-null   object        
 8   location          6735 non-null   object        
 9   transaction_date  9540 non-null   datetime64[ns]
 10  year              9540 non-null   float64       
 11  month             9540 non-null   float64       
 12  day_of_week       9540 non-null   object        
dtypes: datetime64[ns](1), float64(6), object(6)
memory usage: 1015.8+ KB


Esta salida permite comprobar si la conversión de tipos se aplicó correctamente.

Las columnas numéricas deberían aparecer como `float64`, porque contienen valores numéricos y también pueden contener `NaN`. La columna `transaction_date` debería aparecer como `datetime64[ns]`.

Las columnas de texto o categorías pueden seguir apareciendo como `object`. Eso no es necesariamente un problema: un producto, una ubicación o un método de pago son datos categóricos, no magnitudes numéricas.

La validación de tipos es importante porque muchas operaciones posteriores dependen de esta estructura. Si una columna de importes siguiera como texto, los cálculos y resúmenes numéricos podrían fallar o producir resultados incorrectos.

## Revisar valores faltantes finales

Después de preparar el dataset, conviene revisar cuántos valores faltantes quedan.

Es importante recordar que el objetivo de esta limpieza no fue eliminar todos los faltantes. En algunos casos decidimos conservarlos porque no teníamos una justificación suficiente para completarlos. En otros casos, algunos valores problemáticos pasaron a ser `NaN` o `NaT` durante la conversión de tipos.

Por eso, la presencia de faltantes en `df_preparado` no significa automáticamente que la limpieza haya fallado. Lo importante es saber dónde están, cuántos son y qué representan.

In [55]:
df_preparado.isna().sum()

,0
transaction_id,0
item,333
quantity,479
price_per_unit,533
total_spent,502
calculated_total,994
estado_total,0
payment_method,2579
location,3265
transaction_date,460


Esta salida muestra la cantidad de valores faltantes por columna en el dataset preparado.

Algunas columnas pueden tener más faltantes que en el dataset original. Por ejemplo, las columnas numéricas pueden haber recibido nuevos `NaN` cuando valores como `"UNKNOWN"` o `"ERROR"` no pudieron convertirse a número. La columna `transaction_date` también puede tener más faltantes porque algunos textos no pudieron interpretarse como fechas y quedaron como `NaT`.

Podemos construir una tabla con cantidad y porcentaje de faltantes para interpretar mejor el estado final.

In [56]:
diagnostico_faltantes_final = pd.DataFrame({
    "faltantes": df_preparado.isna().sum(),
    "porcentaje": df_preparado.isna().mean() * 100
})

diagnostico_faltantes_final.sort_values("porcentaje", ascending=False)

,faltantes,porcentaje
location,3265,32.65
payment_method,2579,25.79
calculated_total,994,9.94
price_per_unit,533,5.33
total_spent,502,5.02
quantity,479,4.79
year,460,4.60
month,460,4.60
transaction_date,460,4.60
day_of_week,460,4.60


Esta tabla permite priorizar la revisión.

Una columna con muchos faltantes puede limitar ciertos análisis. Por ejemplo, si `location` tiene muchos valores ausentes, los análisis por ubicación deberán interpretarse con cuidado. Si `transaction_date` tiene valores faltantes o no interpretables, algunas transacciones no podrán incluirse en análisis temporales.

La validación posterior no exige resolver todos los problemas en este momento. Su función es dejar claro qué tan preparado está el dataset y qué limitaciones todavía deben considerarse.

## Revisar duplicados

Otra parte importante de la validación posterior consiste en revisar si el dataset preparado contiene duplicados.

En este flujo de limpieza no eliminamos duplicados. Sin embargo, conviene comprobar si existen filas repetidas o identificadores de transacción duplicados.

Primero revisamos duplicados completos.

In [57]:
df_preparado.duplicated().sum()

np.int64(0)

Este conteo indica cuántas filas completas están duplicadas en `df_preparado`.

Si el resultado es `0`, significa que no hay filas exactamente iguales en todas sus columnas.

También podemos revisar duplicados según el identificador de transacción. En este dataset, `transaction_id` debería identificar cada operación.

In [58]:
df_preparado.duplicated(subset=["transaction_id"]).sum()

np.int64(0)

Este conteo indica si hay identificadores de transacción repetidos.

La revisión por identificador es importante porque, aunque dos filas no sean exactamente iguales, un identificador repetido podría indicar una carga duplicada o un problema en el registro.

Si aparecieran identificadores duplicados, deberíamos observar esas filas antes de decidir qué hacer.

In [59]:
df_preparado[
    df_preparado.duplicated(subset=["transaction_id"], keep=False)
].head()

,transaction_id,item,quantity,price_per_unit,total_spent,calculated_total,estado_total,payment_method,location,transaction_date,year,month,day_of_week


Esta última vista muestra las filas involucradas si existen identificadores repetidos.

Si la salida está vacía, significa que no se encontraron casos para revisar bajo ese criterio.

Validar duplicados ayuda a confirmar que el dataset preparado no contiene repeticiones evidentes que puedan afectar conteos, importes o análisis posteriores.

## Revisar categorías problemáticas

Después de la limpieza, también conviene revisar las columnas categóricas.

En nuestro flujo reemplazamos `"ERROR"` por `"UNKNOWN"` en las columnas `item`, `payment_method` y `location`. Esa decisión agrupó valores problemáticos dentro de una categoría común de dato desconocido.

Ahora necesitamos verificar si ese reemplazo se aplicó correctamente y qué categorías siguen presentes.

In [60]:
columnas_categoricas = [
    "item",
    "payment_method",
    "location"
]

for columna in columnas_categoricas:
    print(f"\nColumna: {columna}")
    print("-" * 40)
    print(df_preparado[columna].value_counts(dropna=False))


Columna: item
----------------------------------------
item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      636
NaN          333
Name: count, dtype: int64

Columna: payment_method
----------------------------------------
payment_method
NaN               2579
Digital Wallet    2291
Credit Card       2273
Cash              2258
UNKNOWN            599
Name: count, dtype: int64

Columna: location
----------------------------------------
location
NaN         3265
Takeaway    3022
In-store    3017
UNKNOWN      696
Name: count, dtype: int64


Esta revisión permite observar las categorías finales de cada columna.

Si el reemplazo se aplicó correctamente, `"ERROR"` ya no debería aparecer como categoría separada. En cambio, sus casos deberían estar agrupados dentro de `"UNKNOWN"`.

Podemos verificarlo de manera más directa:

In [61]:
for columna in columnas_categoricas:
    cantidad_error = (df_preparado[columna] == "ERROR").sum()
    print(f"{columna}: {cantidad_error} valores ERROR")

item: 0 valores ERROR
payment_method: 0 valores ERROR
location: 0 valores ERROR


Si todos los conteos dan `0`, significa que no quedan valores `"ERROR"` en esas columnas.

Esto no significa que las columnas categóricas estén perfectas. Todavía pueden quedar valores `"UNKNOWN"` y valores faltantes reales. Lo importante es que ahora esos casos están representados de manera más consistente.

Validar categorías problemáticas ayuda a confirmar que las decisiones de limpieza se aplicaron como esperábamos.

## Revisar rangos numéricos

Después de convertir columnas numéricas, conviene revisar sus rangos.

Esta validación permite detectar valores sospechosos. Por ejemplo, en un dataset de ventas podríamos prestar atención a cantidades negativas, precios iguales a cero, importes negativos o valores extremadamente altos.

En nuestro dataset preparado, las columnas numéricas principales son:

```text
quantity
price_per_unit
total_spent
calculated_total
```

Podemos comenzar con un resumen estadístico.


In [62]:
columnas_numericas_validar = [
    "quantity",
    "price_per_unit",
    "total_spent",
    "calculated_total"
]

df_preparado[columnas_numericas_validar].describe()

,quantity,price_per_unit,total_spent,calculated_total
count,9521.000000,9467.000000,9498.000000,9006.000000
mean,3.028463,2.949984,8.924352,8.936098
std,1.419007,1.278450,6.009919,5.994375
min,1.000000,1.000000,1.000000,1.000000
25%,2.000000,2.000000,4.000000,4.000000
50%,3.000000,3.000000,8.000000,8.000000
75%,4.000000,4.000000,12.000000,12.000000
max,5.000000,5.000000,25.000000,25.000000


La salida de `describe()` nos permite revisar valores mínimos, máximos, promedios y cuartiles.

En esta etapa no buscamos interpretar cada número en profundidad. El objetivo es detectar señales generales: valores negativos, ceros inesperados, importes demasiado grandes o rangos que no tengan sentido para el contexto.

También podemos revisar explícitamente si hay cantidades o importes negativos.

In [63]:
for columna in columnas_numericas_validar:
    cantidad_negativos = (df_preparado[columna] < 0).sum()
    print(f"{columna}: {cantidad_negativos} valores negativos")

quantity: 0 valores negativos
price_per_unit: 0 valores negativos
total_spent: 0 valores negativos
calculated_total: 0 valores negativos


Si los conteos son `0`, significa que no se detectaron valores negativos en esas columnas.

También podemos revisar valores iguales a cero. En algunos contextos, un precio o un total igual a cero podría ser válido, por ejemplo una promoción o una devolución. En otros contextos, podría indicar un error de carga. Por eso no debemos corregirlos automáticamente, pero sí registrarlos si aparecen.

In [64]:
for columna in columnas_numericas_validar:
    cantidad_ceros = (df_preparado[columna] == 0).sum()
    print(f"{columna}: {cantidad_ceros} valores iguales a cero")

quantity: 0 valores iguales a cero
price_per_unit: 0 valores iguales a cero
total_spent: 0 valores iguales a cero
calculated_total: 0 valores iguales a cero


Esta revisión ayuda a detectar posibles valores sospechosos.

La validación de rangos numéricos no implica eliminar o modificar datos de inmediato. Primero necesitamos identificar los casos, luego decidir si realmente son errores y recién después aplicar alguna transformación si corresponde.

En un proceso de limpieza responsable, los valores extremos o inesperados se revisan antes de ser corregidos.

## Revisar fechas válidas

Después de convertir una columna a formato temporal, también conviene validar sus fechas.

En nuestro dataset preparado, la columna temporal principal es `transaction_date`.

Durante la conversión, algunos valores quedaron como `NaT`, porque estaban faltantes o no pudieron interpretarse como fechas. Ahora vamos a revisar cuántos casos hay y qué rango temporal cubren las fechas válidas.

In [65]:
df_preparado["transaction_date"].isna().sum()

np.int64(460)

Este conteo muestra cuántas filas no tienen una fecha válida.

Esas filas pueden seguir siendo útiles para otros análisis, pero no podrán ubicarse correctamente en una línea de tiempo.

Ahora revisemos la fecha mínima y la fecha máxima del dataset preparado.

In [66]:
fecha_minima = df_preparado["transaction_date"].min()
fecha_maxima = df_preparado["transaction_date"].max()

print("Fecha mínima:")
print(fecha_minima)

print()

print("Fecha máxima:")
print(fecha_maxima)

Fecha mínima:
2023-01-01 00:00:00

Fecha máxima:
2023-12-31 00:00:00


Esta revisión permite conocer el período cubierto por el dataset.

También ayuda a detectar fechas fuera de rango. Por ejemplo, si esperáramos trabajar con datos de 2023 y apareciera una fecha de 2099, deberíamos revisarla con atención.

Podemos observar cuántas transacciones válidas hay por año.

In [67]:
df_preparado["year"].value_counts(dropna=False).sort_index()

,count
year,
2023.0,9540
NaN,460


Esta salida permite confirmar si las fechas válidas se concentran en un año o si el dataset contiene registros de varios años.

Los valores faltantes en `year` corresponden a filas cuya `transaction_date` quedó como `NaT`.

La validación temporal no busca corregir fechas automáticamente. Su objetivo es comprobar si el rango de fechas tiene sentido y registrar cuántas filas no pueden participar de análisis temporales.

## Revisar consistencia interna

Una parte importante de la validación posterior consiste en revisar si ciertas columnas son coherentes entre sí.

En este dataset existe una relación esperada:

```text
total_spent = quantity × price_per_unit
```

Durante la limpieza creamos la columna `calculated_total` y luego la columna `estado_total`, que resume si el total registrado coincide con el total calculado.

Ahora podemos revisar el resultado de esa validación.

In [68]:
df_preparado["estado_total"].value_counts(dropna=False)

,count
estado_total,
coincide,8544
sin_datos_suficientes,1456


Esta salida muestra el resultado de la validación interna entre `quantity`, `price_per_unit`, `total_spent` y `calculated_total`.

En este caso aparecen dos estados:

```text
coincide
sin_datos_suficientes
```

No aparecen casos `no_coincide`. Esto significa que, entre las filas que tienen datos suficientes para hacer la comparación, el total registrado coincide con el total calculado.

Las filas clasificadas como `sin_datos_suficientes` no indican una inconsistencia entre importes. Indican que falta alguno de los datos necesarios para validar la relación.

En otro dataset, o en otra versión de este archivo, podrían aparecer casos `no_coincide`. Si aparecieran, convendría observarlos con detalle.

In [69]:
df_preparado[df_preparado["estado_total"] == "no_coincide"][
    [
        "quantity",
        "price_per_unit",
        "total_spent",
        "calculated_total",
        "estado_total"
    ]
].head(10)

,quantity,price_per_unit,total_spent,calculated_total,estado_total


Si esta salida está vacía, significa que no hay filas clasificadas como `no_coincide`.

Eso no significa que todos los registros estén completos. Las filas con `sin_datos_suficientes` indican que falta alguno de los datos necesarios para validar la relación.

Podemos observar algunos de esos casos:

In [70]:
df_preparado[df_preparado["estado_total"] == "sin_datos_suficientes"][
    [
        "quantity",
        "price_per_unit",
        "total_spent",
        "calculated_total",
        "estado_total"
    ]
].head(10)

,quantity,price_per_unit,total_spent,calculated_total,estado_total
2,4.0,1.0,NaN,4.0,sin_datos_suficientes
20,NaN,4.0,20.0,NaN,sin_datos_suficientes
25,3.0,4.0,NaN,12.0,sin_datos_suficientes
31,2.0,1.0,NaN,2.0,sin_datos_suficientes
42,2.0,1.5,NaN,3.0,sin_datos_suficientes
55,NaN,1.0,2.0,NaN,sin_datos_suficientes
56,5.0,NaN,15.0,NaN,sin_datos_suficientes
57,NaN,3.0,3.0,NaN,sin_datos_suficientes
65,3.0,NaN,NaN,NaN,sin_datos_suficientes
66,NaN,3.0,6.0,NaN,sin_datos_suficientes


Esta revisión nos permite diferenciar dos situaciones:

```text
no_coincide              → hay datos suficientes, pero el total registrado no coincide con el calculado
sin_datos_suficientes    → no hay información suficiente para validar la relación
```

Esa diferencia es importante. No debemos interpretar todas las filas no coincidentes o incompletas como el mismo tipo de problema.

La validación de consistencia interna ayuda a detectar errores más profundos que no siempre aparecen al revisar tipos de datos o valores faltantes por separado.


## Construir una tabla resumen de validación

Después de revisar distintos aspectos del dataset preparado, podemos construir una tabla resumen.

Esta tabla no reemplaza las revisiones detalladas, pero ayuda a reunir en un solo lugar algunas señales importantes del estado final del dataset.

Podemos incluir información como:

```text
cantidad de filas
cantidad de columnas
duplicados completos
identificadores duplicados
faltantes en columnas clave
fechas no válidas
casos sin datos suficientes para validar el total
casos donde el total no coincide
```

Construir este tipo de resumen es útil porque permite documentar rápidamente el resultado de la limpieza.


In [71]:
resumen_validacion = pd.DataFrame({
    "aspecto": [
        "filas",
        "columnas",
        "duplicados_completos",
        "transaction_id_duplicados",
        "faltantes_item",
        "faltantes_quantity",
        "faltantes_price_per_unit",
        "faltantes_total_spent",
        "faltantes_payment_method",
        "faltantes_location",
        "fechas_no_validas",
        "totales_sin_datos_suficientes",
        "totales_no_coinciden"
    ],
    "valor": [
        df_preparado.shape[0],
        df_preparado.shape[1],
        df_preparado.duplicated().sum(),
        df_preparado.duplicated(subset=["transaction_id"]).sum(),
        df_preparado["item"].isna().sum(),
        df_preparado["quantity"].isna().sum(),
        df_preparado["price_per_unit"].isna().sum(),
        df_preparado["total_spent"].isna().sum(),
        df_preparado["payment_method"].isna().sum(),
        df_preparado["location"].isna().sum(),
        df_preparado["transaction_date"].isna().sum(),
        (df_preparado["estado_total"] == "sin_datos_suficientes").sum(),
        (df_preparado["estado_total"] == "no_coincide").sum()
    ]
})

resumen_validacion

,aspecto,valor
0,filas,10000
1,columnas,13
2,duplicados_completos,0
3,transaction_id_duplicados,0
4,faltantes_item,333
5,faltantes_quantity,479
6,faltantes_price_per_unit,533
7,faltantes_total_spent,502
8,faltantes_payment_method,2579
9,faltantes_location,3265


Esta tabla resume varios controles realizados durante la validación.

Por ejemplo, permite ver rápidamente si se conservaron las filas, si hay duplicados, cuántos faltantes quedan en columnas importantes, cuántas fechas no son válidas y qué resultado tuvo la validación del total.

En un proyecto real, una tabla como esta puede servir como registro del estado final del dataset preparado.

También ayuda a comunicar que limpiar datos no significa necesariamente dejar todo en cero. Puede quedar información faltante o desconocida, pero lo importante es que esos problemas estén detectados, representados correctamente y documentados.

## Qué problemas quedan pendientes

Después de validar el dataset preparado, podemos identificar qué aspectos todavía requieren atención.

La validación nos permite ver que el dataset está mejor estructurado que al comienzo: tiene nombres de columnas consistentes, tipos de datos más adecuados, una columna temporal convertida, valores categóricos problemáticos parcialmente unificados y columnas nuevas para validar relaciones internas.

Sin embargo, eso no significa que el dataset esté perfecto.

Todavía pueden quedar valores faltantes en columnas importantes, como `payment_method`, `location`, `quantity`, `price_per_unit`, `total_spent` o `transaction_date`. También pueden quedar valores `"UNKNOWN"` en columnas categóricas, que representan información desconocida o no confiable.

Estos problemas no necesariamente deben resolverse todos de la misma manera. Por ejemplo, una fila sin `location` puede seguir sirviendo para analizar productos vendidos o totales de venta. En cambio, una fila sin `quantity` o sin `price_per_unit` no permite calcular correctamente el importe esperado.

Por eso, después de validar, conviene separar dos ideas:

```text
problemas tratados
problemas pendientes
```

Los problemas tratados son aquellos sobre los que ya tomamos una decisión concreta. Por ejemplo, convertimos tipos, unificamos `"ERROR"` dentro de `"UNKNOWN"` y creamos una validación de totales.

Los problemas pendientes son aquellos que decidimos conservar para no aplicar una corrección apresurada. Por ejemplo, no imputamos automáticamente todas las ubicaciones faltantes ni eliminamos todas las filas incompletas.

En un proyecto real, esta distinción es muy importante. Un dataset preparado no es necesariamente un dataset sin problemas. Es un dataset cuyos problemas fueron identificados, tratados cuando correspondía y documentados cuando quedaron pendientes.


## Errores frecuentes al validar una limpieza

Un error frecuente después de limpiar un dataset es asumir que, porque el código se ejecutó sin errores, el dataset ya quedó listo para cualquier análisis.

Que una celda no produzca errores no significa necesariamente que la limpieza haya sido correcta. Podemos haber convertido tipos, reemplazado valores o creado columnas nuevas, pero todavía necesitamos verificar si esas transformaciones produjeron el resultado esperado.

Otro error común es revisar solo una parte del dataset. Por ejemplo, podríamos mirar los tipos de datos finales y confirmar que las columnas numéricas quedaron como `float64`, pero olvidarnos de revisar cuántos valores se convirtieron en `NaN`. También podríamos verificar que no hay duplicados completos, pero no revisar si hay identificadores repetidos.

También es un error esperar que un dataset preparado no tenga ningún valor faltante. En muchos casos, una limpieza responsable no completa todos los datos ausentes. A veces es mejor conservar un faltante que inventar un valor sin fundamento.

Otro punto importante es no confundir valores desconocidos con valores válidos. Una categoría como `"UNKNOWN"` puede ser útil para señalar que el dato no está disponible, pero no debería interpretarse como una categoría común del negocio. No es una ubicación real ni un método de pago real; es una marca de información desconocida.

También debemos tener cuidado con las columnas nuevas. Crear `calculated_total`, `estado_total`, `year`, `month` o `day_of_week` puede enriquecer el dataset, pero esas columnas dependen de la calidad de las columnas originales. Si faltan datos en `quantity`, `price_per_unit` o `transaction_date`, las columnas derivadas también pueden quedar incompletas.

Una rutina razonable de validación posterior podría incluir:

```text
comparar dimensiones antes y después
revisar tipos de datos finales
contar valores faltantes
revisar duplicados completos y por identificador
verificar categorías problemáticas
revisar rangos numéricos
validar fechas
comprobar relaciones internas
documentar problemas pendientes
```

La validación posterior no busca declarar que el dataset es perfecto. Busca dejar claro qué tan confiable es, qué transformaciones se aplicaron y qué limitaciones siguen presentes.

## Resumen del capítulo

En este capítulo trabajamos con la validación posterior a la limpieza.

Partimos de una idea central: limpiar un dataset no alcanza. Después de aplicar transformaciones, necesitamos comprobar si el resultado quedó en mejores condiciones para el análisis y qué problemas siguen presentes.

Primero reconstruimos una versión preparada del dataset. Conservamos `df_original` como punto de partida y creamos `df_preparado` como versión transformada. Esa versión incluyó nombres de columnas estandarizados, columnas numéricas convertidas, fecha convertida, valores categóricos problemáticos unificados parcialmente, columnas calculadas y columnas de validación.

Luego comparamos las dimensiones del dataset original y del dataset preparado:

```python
df_original.shape
df_preparado.shape
```

Esta comparación permitió verificar si se conservaron las filas y si cambió la cantidad de columnas.

Después revisamos los tipos de datos finales usando `info()`. Esta validación permitió comprobar que columnas como `quantity`, `price_per_unit`, `total_spent` y `calculated_total` quedaron como numéricas, mientras que `transaction_date` quedó como columna temporal.

También revisamos los valores faltantes finales:

```python
df_preparado.isna().sum()
```

y construimos una tabla con cantidad y porcentaje de faltantes por columna. Esta revisión mostró que el dataset preparado todavía conserva valores faltantes, lo cual no significa necesariamente que la limpieza haya fallado. En varios casos, esos faltantes representan información ausente o valores problemáticos que fueron convertidos a `NaN` o `NaT`.

Más adelante revisamos duplicados completos y duplicados por `transaction_id`:

```python
df_preparado.duplicated().sum()
df_preparado.duplicated(subset=["transaction_id"]).sum()
```

Esta validación permitió comprobar si el dataset preparado contenía registros repetidos o identificadores duplicados.

También revisamos categorías problemáticas. Verificamos que `"ERROR"` ya no apareciera como categoría separada en `item`, `payment_method` y `location`, porque durante el flujo de limpieza habíamos decidido agrupar esos casos dentro de `"UNKNOWN"`.

Luego revisamos rangos numéricos usando `describe()` y conteos de valores negativos o iguales a cero. Esta validación ayuda a detectar valores sospechosos que podrían requerir una revisión posterior.

También validamos fechas. Revisamos cuántas fechas quedaron como faltantes o no interpretables, cuál fue el rango temporal del dataset y cómo se distribuyeron los años registrados.

Después revisamos la consistencia interna entre `quantity`, `price_per_unit`, `total_spent` y `calculated_total` mediante la columna `estado_total`. Esta columna permitió distinguir entre casos que coinciden, casos que no coinciden y casos sin datos suficientes para validar la relación.

Finalmente construimos una tabla resumen de validación, reuniendo en un solo lugar varios controles importantes: cantidad de filas, columnas, duplicados, faltantes, fechas no válidas y resultados de la validación de totales.

La idea principal de este capítulo fue:

```text
Validar después de limpiar permite comprobar qué mejoró, qué problemas siguen presentes y qué limitaciones deben documentarse.
```

Un dataset preparado no es necesariamente un dataset perfecto. Es un dataset cuyo estado conocemos mejor: sabemos qué transformaciones se aplicaron, qué problemas fueron tratados y qué aspectos todavía requieren cuidado.


## Próximo paso

Con este capítulo cerramos la parte técnica de la etapa de preparación y limpieza de datos.

Antes de pasar al siguiente eje temático, conviene detenernos y hacer un balance de lo trabajado hasta aquí. Podemos revisar qué aprendimos, cómo se conectan las herramientas entre sí y qué significa, en la práctica, dejar un dataset listo para analizar.

Ese cierre nos va a permitir ordenar el recorrido completo: desde el diagnóstico inicial hasta la construcción y validación de una versión preparada del dataset.
